In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Generate sample data with seed(0)
np.random.seed(0)
X = 2 * np.random.rand(100, 1)  # 100 random values between 0 and 2
y = 3 * X.flatten() + 4 + np.random.randn(100)  # Linear relation with noise

# Train and record step-by-step history
n_samples, n_features = X.shape
weights = np.zeros(n_features)
bias = 0
learning_rate = 0.01
n_epochs = 1000

step_history = []

for epoch in range(n_epochs):
    w_before = weights.copy()
    b_before = bias

    # Step 1: Predict
    y_predicted = np.dot(X, weights) + bias

    # Step 2: Calculate gradients
    dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
    db = (1 / n_samples) * np.sum(y_predicted - y)

    # Step 3: Update parameters
    weights = weights - learning_rate * dw
    bias = bias - learning_rate * db

    # Step 4: Calculate loss
    loss = np.mean((y_predicted - y) ** 2)

    # Store error per sample
    errors = y_predicted - y

    step_history.append({
        'epoch': epoch,
        'w_before': w_before[0],
        'b_before': b_before,
        'y_predicted': y_predicted.copy(),
        'errors': errors.copy(),
        'dw': dw[0],
        'db': db,
        'w_after': weights[0],
        'b_after': bias,
        'loss': loss,
    })

# Select which epochs to animate
# First 10: 1-10, then 20,30,...,100, then 200,300,...,1000
selected_epochs = (
    list(range(0, 10)) +
    list(range(19, 100, 10)) +
    list(range(199, 1000, 100))
)

# 6 steps per epoch: data, predict, gradients, update, loss, summary
steps_per_epoch = 6
frames = []
for ep in selected_epochs:
    for step in range(steps_per_epoch):
        frames.append((ep, step))

# Set up figure
fig = plt.figure(figsize=(16, 10))

# Left: scatter + regression line
ax1 = fig.add_axes([0.05, 0.42, 0.42, 0.53])
ax1.scatter(X, y, color='blue', alpha=0.5, edgecolors='k', s=20)
reg_line, = ax1.plot([], [], 'r-', lw=2, label='Prediction line')
ax1.set_xlim(-0.1, 2.1)
ax1.set_ylim(min(y) - 1, max(y) + 1)
ax1.set_xlabel('X')
ax1.set_ylabel('y')
ax1.set_title('Linear Regression Training')
ax1.grid(True)
ax1.legend(loc='lower right')

# Right: loss curve
ax2 = fig.add_axes([0.55, 0.42, 0.40, 0.53])
ax2.set_xlim(0, n_epochs)
ax2.set_ylim(0, step_history[0]['loss'] * 1.1)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Loss')
ax2.set_title('Training Loss')
ax2.grid(True)
loss_line, = ax2.plot([], [], 'g-', lw=2)

# Bottom: code display area
ax_code = fig.add_axes([0.05, 0.02, 0.90, 0.37])
ax_code.set_xlim(0, 1)
ax_code.set_ylim(0, 1)
ax_code.axis('off')

# Background for code area
code_bg = plt.Rectangle((0, 0), 1, 1, transform=ax_code.transAxes,
                          facecolor='#1e1e1e', edgecolor='#444', lw=2)
ax_code.add_patch(code_bg)

code_text = ax_code.text(0.02, 0.95, '', transform=ax_code.transAxes,
                          fontsize=9.5, fontfamily='monospace', color='#d4d4d4',
                          verticalalignment='top', linespacing=1.5)

X_plot = np.linspace(-0.1, 2.1, 100).reshape(-1, 1)
X_flat = X.flatten()

def init():
    reg_line.set_data([], [])
    loss_line.set_data([], [])
    code_text.set_text('')
    return reg_line, loss_line, code_text

def animate(frame_idx):
    ep, step = frames[frame_idx]
    h = step_history[ep]

    # Update regression line with current weights
    if step <= 2:
        y_plot = X_plot.flatten() * h['w_before'] + h['b_before']
    else:
        y_plot = X_plot.flatten() * h['w_after'] + h['b_after']
    reg_line.set_data(X_plot.flatten(), y_plot)

    # Update loss curve
    ep_range = list(range(ep + 1))
    losses = [step_history[e]['loss'] for e in ep_range]
    loss_line.set_data(ep_range, losses)

    header = f"Epoch {ep + 1}/{n_epochs}"

    if step == 0:
        code = (
            f"{header}\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"  # Feed all 100 samples into the model (showing first 5)\n"
            f"  X = [{X_flat[0]:.4f}, {X_flat[1]:.4f}, {X_flat[2]:.4f}, {X_flat[3]:.4f}, {X_flat[4]:.4f}, ...] (100 values)\n"
            f"  y = [{y[0]:.4f}, {y[1]:.4f}, {y[2]:.4f}, {y[3]:.4f}, {y[4]:.4f}, ...] (100 values)\n\n"
            f"  # Current Parameters\n"
            f"  W = {h['w_before']:.6f},  B = {h['b_before']:.6f}"
        )
    elif step == 1:
        code = (
            f"{header}\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"  # Predict for all 100 samples (showing first 5)\n"
            f"  X          = [{X_flat[0]:.4f}, {X_flat[1]:.4f}, {X_flat[2]:.4f}, {X_flat[3]:.4f}, {X_flat[4]:.4f}, ...]\n"
            f"  y_actual   = [{y[0]:.4f}, {y[1]:.4f}, {y[2]:.4f}, {y[3]:.4f}, {y[4]:.4f}, ...]\n\n"
            f"▶ y_predicted = np.dot(X, W) + B    →  W * X + B\n"
            f"  y_pred     = [{h['y_predicted'][0]:.4f}, {h['y_predicted'][1]:.4f}, {h['y_predicted'][2]:.4f}, {h['y_predicted'][3]:.4f}, {h['y_predicted'][4]:.4f}, ...]\n"
            f"  error      = [{h['errors'][0]:.4f}, {h['errors'][1]:.4f}, {h['errors'][2]:.4f}, {h['errors'][3]:.4f}, {h['errors'][4]:.4f}, ...]"
        )
    elif step == 2:
        code = (
            f"{header}\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"  # Compute gradients using all 100 errors (showing first 5)\n"
            f"  y_pred     = [{h['y_predicted'][0]:.4f}, {h['y_predicted'][1]:.4f}, {h['y_predicted'][2]:.4f}, {h['y_predicted'][3]:.4f}, {h['y_predicted'][4]:.4f}, ...]\n"
            f"  error      = [{h['errors'][0]:.4f}, {h['errors'][1]:.4f}, {h['errors'][2]:.4f}, {h['errors'][3]:.4f}, {h['errors'][4]:.4f}, ...]\n\n"
            f"▶ dw = (1/100) * np.dot(X.T, (y_predicted - y))  →  dw = {h['dw']:.6f}\n"
            f"▶ db = (1/100) * np.sum(y_predicted - y)          →  db = {h['db']:.6f}"
        )
    elif step == 3:
        code = (
            f"{header}\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"  dw = {h['dw']:.6f},  db = {h['db']:.6f}\n\n"
            f"▶ self.weights -= 0.01 * dw\n"
            f"    W: {h['w_before']:.6f} - 0.01 * {h['dw']:.6f} = {h['w_after']:.6f}\n"
            f"▶ self.bias    -= 0.01 * db\n"
            f"    B: {h['b_before']:.6f} - 0.01 * {h['db']:.6f} = {h['b_after']:.6f}"
        )
    elif step == 4:
        code = (
            f"{header}\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"  W = {h['w_after']:.6f},  B = {h['b_after']:.6f}\n\n"
            f"▶ loss = np.mean((y_predicted - y) ** 2)    →  mean of all 100 squared errors\n"
            f"  error²     = [{h['errors'][0]**2:.4f}, {h['errors'][1]**2:.4f}, {h['errors'][2]**2:.4f}, {h['errors'][3]**2:.4f}, {h['errors'][4]**2:.4f}, ...]\n"
            f"▶ loss = {h['loss']:.6f}"
        )
    else:
        code = (
            f"{header}  ✓ Complete\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"  Data Fed  → All 100 samples\n"
            f"  Predicted → [{h['y_predicted'][0]:.4f}, {h['y_predicted'][1]:.4f}, {h['y_predicted'][2]:.4f}, {h['y_predicted'][3]:.4f}, {h['y_predicted'][4]:.4f}, ...]\n"
            f"  Error     → [{h['errors'][0]:.4f}, {h['errors'][1]:.4f}, {h['errors'][2]:.4f}, {h['errors'][3]:.4f}, {h['errors'][4]:.4f}, ...]\n"
            f"  Updated   → W = {h['w_after']:.6f},  B = {h['b_after']:.6f}\n"
            f"  Loss      → {h['loss']:.6f}"
        )

    code_text.set_text(code)
    return reg_line, loss_line, code_text

ani = FuncAnimation(
    fig, animate, frames=len(frames),
    init_func=init, interval=400, blit=True, repeat=True
)

plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Generate same data with seed(0)
np.random.seed(0)
X = 2 * np.random.rand(100, 1)
y = 3 * X.flatten() + 4 + np.random.randn(100)

# Compute MSE loss for any (W, B)
def compute_loss(W_val, B_val, X, y):
    y_pred = W_val * X.flatten() + B_val
    return np.mean((y_pred - y) ** 2)

# Record the gradient descent path
n_samples = len(y)
W_path, B_path, L_path = [0.0], [0.0], [compute_loss(0, 0, X, y)]
w, b = 0.0, 0.0
learning_rate = 0.01

for epoch in range(1000):
    y_pred = w * X.flatten() + b
    dw = (1 / n_samples) * np.dot(X.flatten(), (y_pred - y))
    db = (1 / n_samples) * np.sum(y_pred - y)
    w -= learning_rate * dw
    b -= learning_rate * db
    loss = np.mean((y_pred - y) ** 2)
    W_path.append(w)
    B_path.append(b)
    L_path.append(loss)

W_path = np.array(W_path)
B_path = np.array(B_path)
L_path = np.array(L_path)

W_min, B_min, L_min = W_path[-1], B_path[-1], L_path[-1]

# Create loss surface grid
W_grid = np.linspace(-1, 6, 120)
B_grid = np.linspace(-2, 8, 120)
W_mesh, B_mesh = np.meshgrid(W_grid, B_grid)
Z_mesh = np.zeros_like(W_mesh)

for i in range(W_mesh.shape[0]):
    for j in range(W_mesh.shape[1]):
        Z_mesh[i, j] = compute_loss(W_mesh[i, j], B_mesh[i, j], X, y)

# Clip the surface so bowl shape is visible near the bottom
Z_clip = 15.0
Z_mesh_clipped = np.clip(Z_mesh, None, Z_clip)

# Select which epochs to animate
# First 10: 1-10, then 20,30,...,100, then 200,300,...,1000
selected_epochs = (
    list(range(0, 10)) +
    list(range(19, 100, 10)) +
    list(range(199, 1000, 100))
)

# Set up figure
fig = plt.figure(figsize=(16, 6))

# Left: 3D surface
ax3d = fig.add_subplot(121, projection='3d')
ax3d.plot_surface(W_mesh, B_mesh, Z_mesh_clipped, cmap='viridis', alpha=0.5, edgecolor='none')
ax3d.set_xlabel('W (weight)')
ax3d.set_ylabel('B (bias)')
ax3d.set_zlabel('Loss (MSE)')
ax3d.set_title('Gradient Descent on Loss Surface')
ax3d.set_zlim(0, Z_clip)
ax3d.view_init(elev=35, azim=-50)

# Mark the minimum on the surface
ax3d.plot([W_min], [B_min], [L_min], 'g*', markersize=15, zorder=10)

# Path line and current point on 3D
path3d, = ax3d.plot([], [], [], 'r-', lw=2.5)
point3d, = ax3d.plot([], [], [], 'ro', markersize=10, zorder=10)

# Right: contour plot (top-down view)
ax2d = fig.add_subplot(122)
contour = ax2d.contourf(W_mesh, B_mesh, Z_mesh, levels=50, cmap='viridis')
plt.colorbar(contour, ax=ax2d, label='Loss (MSE)')
ax2d.set_xlabel('W (weight)')
ax2d.set_ylabel('B (bias)')
ax2d.set_title('Contour View - Gradient Descent Path')

# Path line and current point on contour
path2d, = ax2d.plot([], [], 'r-', lw=1.5)
point2d, = ax2d.plot([], [], 'ro', markersize=8)
# Mark the minimum and start
ax2d.plot(W_min, B_min, 'g*', markersize=15, markeredgecolor='black', label=f'Minimum ({W_min:.2f}, {B_min:.2f})')
ax2d.plot(0, 0, 'rs', markersize=10, markeredgecolor='black', label='Start (0, 0)')
ax2d.legend(loc='upper right', fontsize=8)

# Info text
info_text = ax2d.text(0.02, 0.02, '', transform=ax2d.transAxes,
                       fontsize=9, fontfamily='monospace',
                       verticalalignment='bottom',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9))

def init():
    path3d.set_data([], [])
    path3d.set_3d_properties([])
    point3d.set_data([], [])
    point3d.set_3d_properties([])
    path2d.set_data([], [])
    point2d.set_data([], [])
    info_text.set_text('')
    return path3d, point3d, path2d, point2d, info_text

def animate(idx):
    ep = selected_epochs[idx]

    # Clip loss values for 3D display
    L_clipped = np.clip(L_path[:ep+1], None, Z_clip)

    # Path up to current epoch
    path3d.set_data(W_path[:ep+1], B_path[:ep+1])
    path3d.set_3d_properties(L_clipped)

    # Current point
    point3d.set_data([W_path[ep]], [B_path[ep]])
    point3d.set_3d_properties([min(L_path[ep], Z_clip)])

    # 2D contour path
    path2d.set_data(W_path[:ep+1], B_path[:ep+1])
    point2d.set_data([W_path[ep]], [B_path[ep]])

    # Distance to minimum
    dist = np.sqrt((W_path[ep] - W_min)**2 + (B_path[ep] - B_min)**2)

    info_text.set_text(
        f'Epoch: {ep + 1}/1000\n'
        f'W = {W_path[ep]:.4f}  (target: {W_min:.4f})\n'
        f'B = {B_path[ep]:.4f}  (target: {B_min:.4f})\n'
        f'Loss = {L_path[ep]:.4f}\n'
        f'Distance to min = {dist:.4f}'
    )

    return path3d, point3d, path2d, point2d, info_text

ani = FuncAnimation(
    fig, animate, frames=len(selected_epochs),
    init_func=init, interval=500, blit=False, repeat=True
)

plt.tight_layout()
plt.close(fig)
HTML(ani.to_jshtml())